In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

: 

: 

In [ ]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

: 

: 

In [ ]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

: 

: 

In [ ]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

: 

: 

In [ ]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

: 

: 

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('llm.env')
openai_client = OpenAI()

: 

: 

In [ ]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

: 

: 

In [ ]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course late. If you want to receive a certificate, make sure you submit your project while submissions are still open.'

: 

: 

In [ ]:
assistant.total_cost()

0.0004965

: 

: 

In [ ]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

: 

: 

In [ ]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want to receive a certificate, make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

: 

: 

In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

: 

: 

In [ ]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

: 

: 

In [ ]:
assistant.reset_usage()

: 

: 

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

: 

: 

In [ ]:
ground_truth_filtered = [rec for rec in ground_truth if rec["document"] in doc_idx]
print(f"Kept {len(ground_truth_filtered)} of {len(ground_truth)} records")

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth_filtered, generate_rag_answer)

Kept 315 of 395 records


  0%|          | 0/315 [00:00<?, ?it/s]

: 

: 

In [ ]:
answers = []

for answer_record in results:
    answers.append(answer_record)

: 

: 

In [ ]:
assistant.total_cost()

0.31515524999999955

: 

: 

In [ ]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

: 

: 

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

: 

: 

In [ ]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

: 

: 

In [ ]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

: 

: 

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv('llm.env')
openai_client = OpenAI()

: 

: 

In [ ]:
rec = answers[0]
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join late. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

: 

: 

In [ ]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

print(prompt)

Question:
Is it okay to join the course late if I just found it now?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join late. If you want a certificate, make sure to submit your project while submissions are still open.


: 

: 

In [ ]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

print(eval_result)

reasoning='The AI answer preserves the key points of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. Semantically equivalent.' score='good'


: 

: 

In [ ]:
calc_price(usage)

{'input_cost': 0.000219, 'output_cost': 0.0002205, 'total_cost': 0.0004395}

: 

: 

In [ ]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

: 

: 

In [ ]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent to the ground truth.', score='good')

: 

: 

In [ ]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

: 

: 

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/315 [00:00<?, ?it/s]

: 

: 

In [ ]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

: 

: 

In [ ]:
evaluations

[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer preserves the key meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project while submissions are still open. It is semantically equivalent.'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer preserves the key point from the ground truth: you can still take the course after the start date, but to receive a certificate you must submit the project while submissions are still open. The added detail about materials being available and starting whenever you want does not change the meaning.'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer preser

: 

: 

In [ ]:
usages

[ResponseUsage(input_tokens=292, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=53, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=345),
 ResponseUsage(input_tokens=314, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=73, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=387),
 ResponseUsage(input_tokens=307, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=54, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=361),
 ResponseUsage(input_tokens=288, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=53, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=341),
 ResponseUsage(input_tokens=351, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=111, output_tokens_de

: 

: 

In [ ]:
calc_total_price(usages)

0.2042625

: 

: 

In [ ]:
df_eval = pd.DataFrame(evaluations)

: 

: 

In [ ]:
df_eval.head()

,question,document,score,reasoning
0,Is it okay to join the course late if I just f...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Can I still take this course even if I missed ...,74eb249bbf,good,The AI answer preserves the key point from the...
2,If I join after the course has already started...,74eb249bbf,good,The AI answer preserves the ground truth: late...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer matches the ground truth: it con...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,good,The AI answer preserves the core requirement t...


: 

: 

In [ ]:
df_eval.score.value_counts()

score
good    290
bad      25
Name: count, dtype: int64

: 

: 

In [ ]:
df_eval.score.value_counts(normalize=True)

score
good    0.920635
bad     0.079365
Name: proportion, dtype: float64

: 

: 

In [ ]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 290/315 = 92.06%


: 

: 

In [ ]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
24,Is peer-review of the capstone project require...,69d122f12e,bad,The AI answer contradicts the ground truth. Th...
33,How will I know when a module is actually read...,96286b4be4,bad,The AI answer does not convey the actual crite...
39,Can you tell me when the course is coming back?,bd31146b0e,bad,The ground truth gives a specific return time:...
58,Is the `OpenAI API Error: Insufficient credits...,f5df151c59,bad,The ground truth says the error can occur when...
63,Which model should I use in chat.completions.c...,152af39a53,bad,The ground truth says the issue is insufficien...


: 

: 

In [ ]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

: 

: 

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

: 

: 

In [ ]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

: 

: 

In [ ]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

: 

: 

In [ ]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

: 

: 

: 

In [ ]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Is it okay to join the course late if I just found it now?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late enrollment join course late found course now registration start late join"}', call_id='call_mdcPT9d8s6Ahefss0zGWDey1', name='search', type='function_call', id='fc_02ff914109191672006a513749bc18819fa939c4b9541340a2', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_mdcPT9d8s6Ahefss0zGWDey1',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your pro

: 

: 

: 

In [ ]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

: 

: 

: 

In [ ]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late join course found it now enroll late add drop policy"}'}]

: 

: 

: 

In [ ]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

: 

: 

: 

In [ ]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_agent': 'Yes — you can still join the course if you just found it.\n\nOne important caveat: if you want a certificate, you’ll need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late join course found it now enroll late add drop policy"}'}],
 'cost': Decimal('0.0010635'),
 'document': '74eb249bbf'}

: 

: 

: 

In [ ]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

: 

: 

: 

In [ ]:
ground_truth_filtered = [rec for rec in ground_truth if rec["document"] in doc_idx]

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth_filtered[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

KeyError: 'c6c2888275'

: 

: 

: 

In [ ]:
df_agent = pd.DataFrame(agent_answers)

: 

: 

: 